In [ ]:
#@title 按這裡開始（先按 ▶）
print('✅ W06 出發！本週目標：自己寫迴圈掃描 k=1..15，把過擬合畫成一張看得懂的圖')
DEMO = ('https://raw.githubusercontent.com/myliao2007/stust-course-1151/'
        'main/ai-intro-pc/data/demo_class.csv')  # ←投影片未含，執行所需：班級資料備援
print('公平比較的第一條規則：一次只改一個參數，其他全部不動。')

# W06　掃描參數，找出過擬合的位置

**先存副本**：`檔案 → 在雲端硬碟中儲存副本`，檔名 `AI導論_W06_學號_姓名`。

### 公平比較的實驗管線

同一份資料（同一個 `random_state`）→ 只改一個參數 → 跑一遍（訓練與測試各打分）
→ 記進實驗表 → 畫成曲線，看出過擬合點。

### 公平比較的四個原則

1. **一次只改一個參數**：兩個一起改，你分不出是誰造成差異
2. **固定 `random_state`**：不固定的話每次切分都不同，分數會跳
3. **訓練測試都要記**：只看測試分數，看不出有沒有過擬合
4. **跑完馬上填記錄表**：不記數字就等於沒做，最後畫不出曲線

## 第 1 格：先讀 W03 的班級資料（任務一要用）

In [ ]:
SHEET_ID = ''    # 貼上老師給的 ID（網址 /d/ 與 /edit 中間那串）；留空白＝用示範資料

import pandas as pd
BASE = 'https://docs.google.com/spreadsheets/d/'
if SHEET_ID.strip():
    df = pd.read_csv(BASE + SHEET_ID + '/export?format=csv')
    # 中文欄名打字麻煩，這一行統一改成英文
    df.columns = ['ts', 'height_cm', 'commute_min', 'phone_hours']
else:                       # ←投影片未含，執行所需
    df = pd.read_csv(DEMO)  # ←投影片未含，執行所需：沒 ID 就讀課程示範資料
print(df.shape)
df.head()

## 任務一：自己切訓練測試集並評估迴歸

兩個 `____` 由你自己寫，**先猜 R² 會是多少再跑**。

**會看到**：兩個 R² 都很低是正常的 —— 通勤時間本來就猜不出手機時數。
測試 R² 甚至可能是負的，代表比直接猜平均還差。

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

d = df[['commute_min', 'phone_hours']].dropna()
X = d[['commute_min']]
y = d['phone_hours']

X_tr, X_te, y_tr, y_te = ____(        # ← ① 切訓練與測試
    X, y, test_size=0.2, random_state=42)

m = LinearRegression().fit(X_tr, y_tr)
print('訓練 R2 =', round(m.score(X_tr, y_tr), 3))
print('測試 R2 =', round(____, 3))    # ← ② 測試集的分數

## 任務二：自己寫迴圈掃 k＝1 到 15

`range` 的範圍與測試分數那一行，**兩個都自己寫**。

**寫對了會看到**：印出 15 列，每列是 `k 訓練分數 測試分數`。
`k=1` 那一列訓練分數一定是 `1.0`，這就是過擬合的徵兆。

In [ ]:
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier

X, y = load_iris(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

ks, tr, te = [], [], []
for k in range(____, ____):         # ← ① 1 到 15
    mdl = KNeighborsClassifier(n_neighbors=k).fit(Xtr, ytr)
    ks.append(k)
    tr.append(mdl.score(Xtr, ytr))
    te.append(____)                 # ← ② 測試集分數
    print(k, round(tr[-1], 3), round(te[-1], 3))

### 實驗記錄表（一邊跑一邊填）

| k 值 | 訓練分數 | 測試分數 | 兩者差距 |
|---|---|---|---|
| 1 | | | |
| 3 | | | |
| 5 | | | |
| 7 | | | |
| 11 | | | |
| 15 | | | |
| 差距最大的 k 是 | | | |
| 測試最高的 k 是 | | | |

---

## 任務三：畫出正確率曲線，找出過擬合

兩條線畫在同一張圖上，**差距最大的地方就是過擬合**。

**會看到**：一張兩條折線的圖（train 圓點、test 方點）與圖例，
以及 `最好的 k = ?`。PNG 存下來一起交。

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(ks, tr, marker='o', label='train')
plt.____(ks, te, marker='s', label='test')  # ← ① 測試曲線
plt.xlabel('k')
plt.ylabel('accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('w06_k.png', dpi=150)
plt.show()

best = ks[te.index(____)]           # ← ② 測試分數最高的
print('最好的 k =', best)

### 從這張曲線要看出三件事

1. **`k=1` 的訓練分數**：一定是 1.0。每一筆的最近鄰居就是它自己，這是過擬合最極端的樣子。
2. **兩條線的距離**：距離越大代表背得越兇。距離開始縮小的地方，就是從背答案轉向抓規律。
3. **測試線的最高點**：那個 k 才是該選的。再往右走界線變模糊，兩條線會一起往下掉。

---

## 任務四：決策樹改深度，印出規則

最後的 `____` 填**你剛剛看到測試分數最高的那個深度**。

**會看到**：三列 depth 的訓練／測試分數，再印出前兩層的判斷規則。
`depth 8` 訓練 1.0、測試卻沒有更好，那就是過擬合的位置。

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

for depth in [1, 3, 8]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=0)
    t.fit(Xtr, ytr)
    print('depth', depth,
          '訓練', round(t.score(Xtr, ytr), 3),
          '測試', round(t.score(Xte, yte), 3))

bt = DecisionTreeClassifier(max_depth=____, random_state=0)
bt.fit(Xtr, ytr)
print(export_text(bt, max_depth=2))

## 任務五（進階）：5 折交叉驗證

KNN 一定要先做尺度縮放，這裡用 pipeline 一起包。

**會看到**：五個分數（有高有低）與一個平均，平均比單次切分可靠得多。

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

pipe = make_pipeline(StandardScaler(),
                     KNeighborsClassifier(n_neighbors=best))

s = cross_val_score(pipe, X, y, cv=____)   # ← ① 幾折
print('五次分數', [round(v, 3) for v in s])
print('平均', round(s.____(), 3))          # ← ② 取平均

## 討論與繳交

先討論三分鐘再打字，答案寫在筆記本最後一格。

1. **討論一：k 從 1 到 15，訓練分數為什麼一路往下？** 提示：k＝1 時，每一筆最近的鄰居就是它自己
2. **討論二：你選哪一個 k？理由是什麼？** 看測試分數最高、而且兩條線已經靠近的位置
3. **要交的東西**：筆記本連結 ＋ `w06_k.png`（要看得到兩條線與圖例）
4. **實驗記錄表八列填完，拍照上傳**：最後兩列（差距最大、測試最高）不能空白

### 常見狀況排除

| 狀況 | 原因 | 怎麼處理 |
|---|---|---|
| 每次分數都不一樣 | `random_state` 沒固定 | 三個模型都加 `random_state=42` |
| `NameError: df` | 還沒讀 W03 的班級資料 | 先跑讀 CSV 那一格，再跑第 1 格 |
| R² 是負的 | 測試集太小又沒關係 | 正常現象，代表比直接猜平均還差 |
| 曲線只有一個點 | 忘了 `append` 進 list | 檢查三行 `append` 有沒有在迴圈內 |
| 圖存出來空白 | `savefig` 寫在 `show` 後面 | 把 `savefig` 移到 `show` 上面一行 |
| KNN 跑很久 | k 掃太多或資料太大 | 先跑 1 到 15，不要一次掃到 100 |
| `export_text` 太長 | 深度太深規則印不完 | 加 `max_depth=2` 只印前兩層 |
| 下課後圖與檔案不見 | 存在執行階段暫存區 | PNG 下載到電腦，筆記本存回硬碟 |

### 延伸挑戰（A 到 C 越後面越難）

- **A 改參數再跑**：把 `test_size` 從 0.2 改成 0.4 再掃一次 k。最好的 k 有沒有換人？
- **B 換一種做法**：把 KNN 換成決策樹，改掃 `max_depth`＝1 到 10，畫出同樣的兩條曲線。
- **C 說出為什麼**：為什麼 KNN 要先做尺度縮放，決策樹卻不用？用班級資料舉一個例子說明。

---

<details>
<summary><b>參考解</b>（真的卡住再打開，先自己試滿 10 分鐘）</summary>

**任務一**

```python
X_tr, X_te, y_tr, y_te = train_test_split(        # ①
    X, y, test_size=0.2, random_state=42)
print('測試 R2 =', round(m.score(X_te, y_te), 3))  # ②
```

**任務二**

```python
for k in range(1, 16):              # ① 要含 15，所以寫到 16
    ...
    te.append(mdl.score(Xte, yte))  # ②
```

**任務三**

```python
plt.plot(ks, te, marker='s', label='test')   # ①
best = ks[te.index(max(te))]                 # ②
```

**任務四**：`max_depth=3` —— 用任務二那張曲線與這一格三列分數挑出來的那個深度。

**任務五**

```python
s = cross_val_score(pipe, X, y, cv=5)   # ①
print('平均', round(s.mean(), 3))        # ②
```

</details>